In [ ]:
import pandas as pd
import numpy as np
import kagglehub
from sklearn.model_selection import train_test_split

from capstone.dataset import build_dataset, dedup_dataset, select_feature_columns
from capstone.prep import prep_data
from capstone.classic import train_classical_model, classical_predict_proba
from capstone.evaluate import evaluate_model
from capstone.dnn import train_dnn

%load_ext autoreload
%autoreload 2


## TODO

In [ ]:
def prepare_experiment(
    raw: dict[str, pd.DataFrame], 
    stratify_by_source : bool, 
    test_size : float = 0.2, 
    random_state : int = 42
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    df = build_dataset(raw)
    df = dedup_dataset(df)

    if stratify_by_source:
        strat_key = df["Label"].astype(str) + "_" + df["Source"]
    else:
        strat_key = df["Label"]
    
    df_train, df_test = train_test_split(df, test_size=test_size, stratify=strat_key, random_state=random_state)

    exclude_cols = ["Source", "Subject", "Body", "Label", "text"]
    candidate_cols = [c for c in df.columns if c not in exclude_cols]
    feature_cols = select_feature_columns(df_train, candidate_cols)

    return df_train, df_test, feature_cols



In [ ]:
# Download the consoldiated spam data set from kaggle; the kaggle API handles local caching
path = kagglehub.dataset_download("nitishabharathi/email-spam-dataset")

raw_data_sets = { 
    'Enron': pd.read_csv(f"{path}/enronSpamSubset.csv"),
    'Spam Assassin': pd.read_csv(f"{path}/completeSpamAssassin.csv"),
    'LingSpam': pd.read_csv(f"{path}/lingSpam.csv")
}

In [ ]:
enron_train, enron_test, enron_features = prepare_experiment(
    { "Enron" : raw_data_sets["Enron"] }, stratify_by_source=False
)
combined_train, combined_test, combined_features = prepare_experiment(
    raw_data_sets, stratify_by_source=True
)

# comfortably above the largest genuine outlier (23,343), well below the corrupted row (3,527,577)
CORRUPTION_THRESHOLD = 100_000  

token_counts = combined_train["text"].str.split().str.len()
combined_train = combined_train[token_counts <= CORRUPTION_THRESHOLD]


## TODO

In [ ]:
%%time
enron_grid = train_classical_model(enron_train, feature_cols=enron_features)
print("Enron only best parameters: ", enron_grid.best_params_)

In [ ]:
%%time
combined_grid = train_classical_model(combined_train, feature_cols=combined_features)
print("Combined best parameters: ", combined_grid.best_params_)

### TODO

In [ ]:
df_full = pd.concat([combined_train, combined_test], ignore_index=True)
df_full = df_full[df_full["Source"] != "Enron"]

enron_predict_proba = classical_predict_proba(enron_grid.best_estimator_)
combined_predict_proba = classical_predict_proba(combined_grid.best_estimator_)

results = { 
    "Enron only" : evaluate_model(enron_predict_proba, enron_test, "Label"),
    "Enron only (isolated test data)" : evaluate_model(enron_predict_proba, df_full, "Label"),
    "Combined": evaluate_model(combined_predict_proba, combined_test, "Label")
}

In [ ]:
for source_name, source_df in combined_test.groupby("Source"):
    results[f"Combined ({source_name})"] = evaluate_model(combined_predict_proba, source_df, "Label")

for name, metrics in results.items():
    print(f"{name}: precision={metrics['precision']:.3f} recall={metrics['recall']:.3f} "
          f"f1={metrics['f1']:.3f} roc_auc={metrics['roc_auc']:.3f}")

### DNN

In [ ]:
%%time
enron_dnn = train_dnn(enron_train, enron_features)

In [ ]:
token_counts = combined_train["text"].str.split().str.len()
print(token_counts.describe())
print("95th percentile:", np.percentile(token_counts, 95))
print("99th percentile:", np.percentile(token_counts, 99))
print("max:", token_counts.max())

In [ ]:
worst_idx = token_counts.idxmax()
print(combined_train.loc[worst_idx, "Source"])
print(combined_train.loc[worst_idx, "text"][:1000])

In [ ]:
combined_train.loc[token_counts.nlargest(10).index, "Source"]

In [ ]:

sample_text = combined_train.loc[7307, "text"]
sample_tokens = sample_text.split()
print(f"chars: {len(sample_text)}, tokens: {len(sample_tokens)}, chars/token: {len(sample_text) / len(sample_tokens):.2f}")

# compare against a typical row
normal_text = combined_train.loc[combined_train.index[0], "text"]
normal_tokens = normal_text.split()
print(f"chars: {len(normal_text)}, tokens: {len(normal_tokens)}, chars/token: {len(normal_text) / len(normal_tokens):.2f}")